# Dataset creation

## Importing artists

The first step involves importing a dataset containing information on Italian artists. Because this data was collected a few years ago, certain data points may no longer reflect the current state of the platform, particularly due to recent modifications in Spotify's genre classification system. Additionally, metrics such as genres, follower counts, and popularity scores were originally gathered using the Spotify API. Updating this information is no longer feasible due to massive API restrictions introduced by Spotify, which have deprecated these data endpoints for standard developer access and now require a Premium account and strict quotas to access even basic data.

In [ ]:
import csv
file_name = "dataset/artists.csv"

artists = []
with open(file_name, 'r', newline='', encoding='utf-16') as file:
        reader = csv.DictReader(file)

        for row in reader:
            if int(row["popularity"]) >= 25:
                artists.append(row)

artists[:2]

[{'id': '582KhTHEVOONNQLmQ5612r',
  'name': 'Calcutta',
  'genres': "['italian pop', 'rome indie']",
  'popularity': '62',
  'followers': '772453',
  'image-url': 'https://i.scdn.co/image/ab6761610000e5ebe6f42fb00b849720936df798'},
 {'id': '7BgEOZ9w3Y4IMShXTMu1nN',
  'name': 'Frah Quintale',
  'genres': "['italian hip hop', 'italian indie pop', 'italian pop', 'milan indie']",
  'popularity': '64',
  'followers': '659919',
  'image-url': 'https://i.scdn.co/image/ab6761610000e5ebb35b656a907a250278c4ecf5'}]

## Downloading lyrics and retrieving genres

Using the list of artist names, we query the Last.fm API to gather the top five user-voted genres, total listener counts, and overall playcounts for each artist. Simultaneously, we use the Genius API to fetch the top 20 most popular song lyrics per artist. To ensure reliability, the data-gathering script incorporates several fault-tolerance mechanisms. It features a checkpoint system to resume from its last position in the event of an interruption. To handle Genius API rate limits, the algorithm automatically rotates through multiple API keys; if all available keys are exhausted, the process terminates gracefully. Furthermore, in the event of a connection error, the script retries the request up to 10 times before skipping the problematic artist. Ultimately, this robust pipeline enabled the successful download of over 25,000 lyrics.

The api key are safely stored inside the .env file.

In [ ]:
import os
import time
import json
import sys
import requests
from dotenv import load_dotenv
from lyricsgenius import Genius

# Load environment variables securely
load_dotenv(override=True) 

# Last.fm API Key
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")

# Load Genius API tokens as a list by splitting the comma-separated string
GENIUS_TOKENS = [t.strip() for t in os.getenv("GENIUS_ACCESS_TOKENS", "").split(",") if t.strip()]
current_token_index = 0

# Verify if at least one Genius token is provided
if not GENIUS_TOKENS:
    print("CRITICAL ERROR: No Genius tokens found in the environment variables. Please check your .env file.")
    sys.exit(1)

# Initialize Genius API with the first available token
genius = Genius(GENIUS_TOKENS[current_token_index])
genius.verbose = False # Turn off status messages to keep the console clean
genius.remove_section_headers = True # Remove section headers (e.g. [Chorus]) from lyrics

# File paths for saving data and execution state
DATA_FILE = 'dataset/tracks_lyrics.json'
INDEX_FILE = 'dataset/index.txt'

tracks = []
artist_index = 0
tracks_saved = 0

os.makedirs('dataset', exist_ok=True)


# Resume mechanism: Check if previous execution files exist to pick up where we left off
if os.path.exists(INDEX_FILE) and os.path.exists(DATA_FILE):
    with open(INDEX_FILE, 'r') as file:
        aux = file.readline().strip()
        if aux:
            artist_index, tracks_saved = [int(x) for x in aux.split("-")]

    with open(DATA_FILE, 'r', encoding="utf-8") as json_file:
        tracks = json.load(json_file)


# Iterate through the artists starting from the saved index
for i in range(artist_index, len(artists)):
    artist = artists[i]

    # Print a status update every 10 artists
    if i % 10 == 0:
        print(f"Artists processed: {i}")
        print(f"Total distinct tracks saved: {tracks_saved}")
        time.sleep(5) # Brief pause to avoid overwhelming the APIs

    # Clean the initial input string to avoid trailing spaces breaking the API
    search_name = artist["name"].strip()


    # Initialize the dictionary structure for the current artist, with artist_id at the top
    artist_data = {
        "artist_id": artist.get("id", ""),
        "artist_name": search_name,
        "genres": [],
        "listeners": 0,
        "playcount": 0,
        "tracks": []
    }


    # Fetch metadata (Genres, Listeners, Playcount) from Last.fm
    lastfm_url = "http://ws.audioscrobbler.com/2.0/"
    lastfm_params = {
        'method': 'artist.getinfo',
        'artist': search_name,
        'api_key': LASTFM_API_KEY,
        'format': 'json'
    }

    for attempt in range(10):
        try:
            response = requests.get(lastfm_url, params=lastfm_params)
            if response.status_code == 200:
                artist_info = response.json().get('artist', {})
                search_name = artist_info.get('name', search_name).strip()
                artist_data["artist_name"] = search_name
                
                tags_data = artist_info.get('tags', {}).get('tag', [])
                artist_data["genres"] = [tag['name'] for tag in tags_data[:5]]
                
                artist_data["listeners"] = int(artist_info.get('stats', {}).get('listeners', 0))
                artist_data["playcount"] = int(artist_info.get('stats', {}).get('playcount', 0))
            else:
                print(f"Last.fm API returned status code {response.status_code} for {search_name}")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} - Last.fm API Error for artist details: {search_name}")
            print(e)
            time.sleep(attempt + 1)

    artist_genius = None


    # Fetch Top 20 tracks and lyrics from Genius with token rotation
    while current_token_index < len(GENIUS_TOKENS):
        success = False
        rate_limit_hit = False
        
        # 10 attempts to fetch data with the current token before skipping artist or rotating token
        for attempt in range(10):
            try:
                artist_genius = genius.search_artist(search_name, max_songs=20, sort="popularity")
                success = True
                break # Successfully fetched data, break the inner retry loop
            except Exception as e:
                error_msg = str(e)
                
                # Check if the error is explicitly a 429 Rate Limit Exceeded
                if "429" in error_msg or "rate limit" in error_msg.lower():
                    print(f"Token index {current_token_index} exhausted (Rate Limit 429 reached).")
                    rate_limit_hit = True
                    break # Break the inner loop immediately to trigger token rotation
                
                else:
                    # Used for temporary network timeouts, 502 Bad Gateway, etc.
                    # Incremental backoff strategy for temporary issues, but do not rotate token for these
                    print(f"Temporary Genius API error (Attempt {attempt + 1}/10) on token index {current_token_index}: {e}")
                    time.sleep(attempt + 1) 
        
        # If the fetch was successful, exit the token rotation loop entirely
        if success:
            break
            
        # If the inner loop was broken due to a true 429 error, switch to the next token
        if rate_limit_hit:
            current_token_index += 1
            if current_token_index < len(GENIUS_TOKENS):
                print(f"Switching to Genius token index {current_token_index}...")
                genius = Genius(GENIUS_TOKENS[current_token_index])
                genius.verbose = False
                genius.remove_section_headers = True
                time.sleep(2) # Short buffer pause before using the fresh token
            else:
                print("CRITICAL: All provided Genius API tokens have been completely exhausted!")
                break
        else:
            # If it failed 10 times but NOT due to a 429, skip this specific artist to prevent an infinite loop lock
            print(f"Skipping artist '{search_name}' due to persistent temporary network/API errors.")
            break

    # If all keys are completely exhausted, terminate the entire script safely
    if current_token_index >= len(GENIUS_TOKENS) and not success:
        print("Script execution stopped. No working Genius API tokens remaining. Progress has been safely saved.")
        break

    # If the artist and songs were successfully found on Genius, validate the match
    if artist_genius is not None:
        target_clean = search_name.lower().replace(" ", "")
        found_clean = artist_genius.name.lower().replace(" ", "")
        
        if target_clean in found_clean or found_clean in target_clean:
            for song in artist_genius.songs:
                artist_data["tracks"].append({
                    "title": song.title,
                    "lyrics": song.lyrics
                })
            tracks_saved += len(artist_genius.songs)
        else:
            print(f"WARNING: Genius artist mismatch! Looked for: '{target_clean}', Found: '{found_clean}'. Skipping tracks.")

    # Append the completed artist data to our main list
    tracks.append(artist_data)

    # Save the JSON data after every artist to prevent data loss
    with open(DATA_FILE, 'w', encoding="utf-8") as json_file:
        json.dump(tracks, json_file, ensure_ascii=False, indent=4)
    
    # Update the index.txt file with the next artist's index and current track count
    with open(INDEX_FILE, 'w') as file:
        file.write(f"{i + 1}-{tracks_saved}")

print(f"Execution finished. Total distinct tracks saved: {tracks_saved}")
print("All data successfully saved.")

Token index 0 exhausted (Rate Limit 429 reached).
Switching to Genius token index 1...
Temporary Genius API error (Attempt 1/10) on token index 1: Request timed out:
HTTPSConnectionPool(host='genius.com', port=443): Read timed out. (read timeout=5)
Artists processed: 10
Total distinct tracks saved: 197
Artists processed: 20
Total distinct tracks saved: 377
Temporary Genius API error (Attempt 1/10) on token index 1: Unexpected response status code: 502. Expected 200 or 204. Response body: error code: 502. Response headers: {'Date': 'Sun, 07 Jun 2026 18:56:49 GMT', 'Content-Type': 'text/plain; charset=UTF-8', 'Content-Length': '15', 'Connection': 'keep-alive', 'CF-Ray': 'a081e1043a9eed8f-MXP', 'Cache-Control': 'private, max-age=0, no-store, no-cache, must-revalidate, post-check=0, pre-check=0', 'Expires': 'Thu, 01 Jan 1970 00:00:01 GMT', 'Proxy-Status': 'Cloudflare-Proxy;error=http_response_incomplete', 'Referrer-Policy': 'same-origin', 'X-Frame-Options': 'SAMEORIGIN', 'Vary': 'accept-en